In [0]:
import sys, os
sys.dont_write_bytecode = True
repo = os.path.abspath("..")
if repo not in sys.path:
    sys.path.insert(0, repo)

In [0]:
for t in ["e2_lkp_areas", "e1_buildings", "e2_buildings", "e1_projects", "e2_projects",
          "e1_developers", "e2_developers", "e1_transactions", "e2_transactions", "transactions_daily"]:
    cols = [c for c in spark.table(f"dld.raw.{t}").columns if not c.startswith("_")]
    print(f"{t} ({len(cols)}):", ", ".join(cols))
    print()

In [0]:
display(spark.table("dld.raw.transactions_daily").limit(5))

In [0]:
for t in ["e1_transactions", "e2_transactions", "transactions_daily", "e1_projects", "e2_projects"]:
    df = spark.table(f"dld.raw.{t}")
    area_cols = [c for c in df.columns if "area" in c.lower()]
    print(t, "->", area_cols)
    if area_cols:
        df.select(area_cols).distinct().limit(5).show(truncate=False)

In [0]:
import pytest
assert pytest.main([f"{repo}/tests", "-q", "-p", "no:cacheprovider"]) == 0

In [0]:
import importlib
from etl.migrate import run_migrations
run_migrations(spark, f"{repo}/sql/ddl")          # applies 100_dims.sql

import etl.build_dims
importlib.reload(etl.build_dims)
from etl.build_dims import build_dims, DIMS
build_dims(spark)

In [0]:
for name, d in DIMS.items():
    k = d["key"]
    r = spark.sql(f"""SELECT count(*) n, count(DISTINCT {k}) k,
                      sum(CASE WHEN {k} IN (-1, -2) THEN 1 ELSE 0 END) sp FROM dld.core.{name}""").first()
    print(name.ljust(14), r.n, "rows |", "keys unique" if r.n == r.k else "DUPLICATE KEYS", "|", r.sp, "special rows")
display(spark.sql("SELECT source, count(*) AS n FROM dld.core.dim_area GROUP BY source ORDER BY n DESC"))

In [0]:
display(spark.sql("""
  WITH d AS (SELECT area_en, count(*) AS rows FROM dld.raw.transactions_daily GROUP BY area_en)
  SELECT d.area_en, d.rows, collect_list(a.area_id) AS matching_area_ids
  FROM d LEFT JOIN dld.core.dim_area a ON lower(trim(d.area_en)) = lower(a.area_name_en)
  GROUP BY d.area_en, d.rows ORDER BY size(matching_area_ids), d.area_en
"""))

In [0]:
display(spark.sql("""
WITH d AS (
  SELECT DISTINCT upper(trim(area_en)) AS name FROM dld.raw.transactions_daily
),
t AS (
  SELECT area_id, area_name_en, master_project_en, project_name_en, nearest_landmark_en FROM dld.raw.e2_transactions
  UNION ALL
  SELECT area_id, area_name_en, master_project_en, project_name_en, nearest_landmark_en FROM dld.raw.e1_transactions
),
hits AS (
  SELECT 'area_name'      AS via, upper(trim(area_name_en))        AS name, area_id FROM t
  UNION ALL SELECT 'master_project', upper(trim(master_project_en)),  area_id FROM t
  UNION ALL SELECT 'project',        upper(trim(project_name_en)),    area_id FROM t
  UNION ALL SELECT 'landmark',       upper(trim(nearest_landmark_en)), area_id FROM t
)
SELECT d.name, h.via, count(h.area_id) AS rows, count(DISTINCT h.area_id) AS area_ids,
       slice(array_sort(collect_set(h.area_id)), 1, 5) AS some_ids
FROM d LEFT JOIN hits h ON h.name = d.name
GROUP BY d.name, h.via
ORDER BY d.name, h.via
"""))